# Find all runs from stability analyses


In [31]:
from pathlib import Path
import re
import pandas as pd
import pickle
from starelib.results import Results
from starelib.util import get_cluster_blobs, reshape_labels_to_3d


In [9]:
cerepet_path = Path("/data/pet/cerepet/derivatives/time_stability")
biograph_path = Path("/data/pet/biograph/derivatives/time_stability")


In [28]:
class Run:
    """ Metadata for one STARE run """
    
    def __init__(self, path):
        self.path = path
        
        # Defaults
        self.results = None
        self.scanner_type = "n/a"
        self.volumes = 0
        self.run_num = 0
        self.subject_id = "n/a"
        self.ki_data = None
        self.log_file = None
        self.command_issued = None
        self.section_starts = list()
        self.cluster_id = dict()
        self.cluster_k = dict()
        self.cluster_centroid = dict()
        self.blob_df = None
        self.blob_ids = None
        self.blob_voxel_counts = None
        self.blob_messages = list()
        
        # Actual
        self.extract_path_metadata()
        self.extract_kis()
        self.extract_clusters()
        self.extract_log()
        self.load_results()
    
    def extract_path_metadata(self):
        pattern = r".*/vols-(?P<volumes>[0-9]+)_run-(?P<run_num>[0-9]+)/.*"
        match = re.match(pattern, str(self.path))
        if "/cerepet/" in str(self.path):
            self.scanner_type = "cerepet"
        elif "/biograph/" in str(self.path):
            self.scanner_type = "biograph"
        else:
            self.scanner_type = "unknown"
        
        if match:
            self.subject_id = self.path.name
            if self.subject_id.startswith("sub-"):
                self.subject_id = self.subject_id[4:]
            self.volumes = int(match.group("volumes"))
            self.run_num = int(match.group("run_num"))
    
    def extract_kis(self):
        """ Load params from any STARE output path. """

        py_param_file = (
            self.path /
            f"sub-{self.subject_id}_final_stare_mean_rate_constants.csv"
        )
        if not py_param_file.exists():
            return None
        
        _param_data = pd.read_csv(py_param_file, index_col=0)
        _param_data = _param_data[['Ki', ]].reset_index(names=['region', ])
        _param_data['subject'] = self.subject_id
        _param_data['source'] = "python"
        _param_data['param'] = 'Ki'
        _param_data['mask_type'] = 'original'
        # TODO: handle alternate cluster overrides instead of just 'original'
        _param_data = _param_data.rename(columns={'Ki': 'value', })
        self.ki_data = _param_data[[
            'subject', 'region', 'source', 'mask_type', 'param', 'value',
        ]]
    
    def extract_clusters(self):
        if self.results is None:
            return None
        
        self.cluster_centroid[1] = None
        for c in self.results.cluster_centroids[1]:
            if c.best_overall:
                self.cluster_centroid[1] = c
        
        self.blob_df, self.blob_ids, self.blob_voxel_counts = get_cluster_blobs(
            reshape_labels_to_3d(
                self.results.cluster_model_fits[1][self.cluster_k[1]].labels_,
                self.cluster_centroid[1].original_shape[:3]
            ),
            label=self.cluster_centroid[1].label,
            verbose=True,
            messages=self.blob_messages,
        )

    
    def extract_log(self):
        command_pattern = re.compile(r".*The command issued: '(.+)'")
        section_pattern = re.compile(r"([0-9]+-[0-9]+-[0-9]+_[0-9]+-[0-9]+).*"
                                     r"Started section '(.+)'.")
        cluster_pattern = re.compile(r".*Centroid ([0-9]+) of k=([0-9]+).*best overall")
        
        for log_path in self.path.glob("stare_pet_*.log"):
            if log_path.exists():
                if self.log_file is None:
                    self.log_file = log_path
                else:
                    print(f"Ignoring a second log file for {self.subject_id}: "
                          f"{str(log_path)}")
        if self.log_file is None:
            print(f"Failed to find a log file at {str(self.path)}")
        else:
            with open(self.log_file, "r") as f:
                for line in f.readlines():
                    match_cmd = command_pattern.search(line)
                    if match_cmd:
                        self.command_issued = match_cmd.group(1)
                    match_sect = section_pattern.search(line)
                    if match_sect:
                        self.section_starts.append(
                            (match_sect.group(1), match_sect.group(2))
                        )
                    match_cluster = cluster_pattern.search(line)
                    if match_cluster:
                        if match_cluster.group(2) == "4":
                            self.cluster_id[2] = int(match_cluster.group(1))
                            self.cluster_k[2] = int(match_cluster.group(2))
                        else:
                            self.cluster_id[1] = int(match_cluster.group(1))
                            self.cluster_k[1] = int(match_cluster.group(2))
    def load_results(self):
        results_path = self.path / f"sub-{self.subject_id}_results.pkl"
        if results_path.exists():
            print(f"loading picked results from {results_path}")
            # self.results = pickle.load(results_path)
        else:
            print(f"Results file does not exist for {self.subject_id}")
        if self.results is None:
            print(f"Failed to load results for {self.subject_id}")
            
    def as_dict(self):
        return {
            "path": str(self.path),
            "subject_id": self.subject_id,
            "scanner_type": self.scanner_type,
            "volumes": self.volumes,
            "run_num": self.run_num,
        }


In [29]:
runs = list()
for data_path in [cerepet_path, biograph_path]:
    for run_path in data_path.glob("vols-22_run-*/NHFDG279"):
        if len(runs) < 5:
            runs.append(Run(run_path))
print(f"Found {len(runs)} runs")

run_df = pd.DataFrame([run.as_dict() for run in runs])


loading picked results from /data/pet/biograph/derivatives/time_stability/vols-22_run-6/NHFDG279/sub-NHFDG279_results.pkl
Failed to load results for NHFDG279
loading picked results from /data/pet/biograph/derivatives/time_stability/vols-22_run-1/NHFDG279/sub-NHFDG279_results.pkl
Failed to load results for NHFDG279
loading picked results from /data/pet/biograph/derivatives/time_stability/vols-22_run-2/NHFDG279/sub-NHFDG279_results.pkl
Failed to load results for NHFDG279
loading picked results from /data/pet/biograph/derivatives/time_stability/vols-22_run-4/NHFDG279/sub-NHFDG279_results.pkl
Failed to load results for NHFDG279
Found 4 runs


----

try out code before adding to loop...


In [23]:
one_run_results = pickle.load(
    open(biograph_path / "vols-22_run-2/NHFDG279/sub-NHFDG279_results.pkl", "rb")
)


In [30]:
one_run = runs[2]


In [32]:
centroid = None
for c in one_run_results.cluster_centroids[1]:
    if c.best_overall:
        centroid = c
labels = one_run_results.cluster_model_fits[1][one_run.cluster_k[1]].labels_
message_list = list()

blob_df, blob_ids, voxel_counts = get_cluster_blobs(
    reshape_labels_to_3d(labels, centroid.original_shape[:3]),
    label=centroid.label,
    verbose=True,
    messages=message_list,
)


In [24]:
for i, run in enumerate(runs):
    if run.subject_id == "NHFDG279":
        print(i, run.path)

63 /data/pet/biograph/derivatives/time_stability/vols-22_run-6/NHFDG279
83 /data/pet/biograph/derivatives/time_stability/vols-22_run-1/NHFDG279
100 /data/pet/biograph/derivatives/time_stability/vols-22_run-2/NHFDG279
113 /data/pet/biograph/derivatives/time_stability/vols-22_run-4/NHFDG279
